# V2 GPU 정밀도 진단과 고정 조합 재실행

첫 실험의 학습·새 평가 결과는 보존합니다. 여기서는 같은 가중치로 ZIP 예측 차이를 재현하고, 원인이 확인될 때만 런타임 정밀도 설정을 수정합니다. 개발 자료에서 선택한 G0_AST_MUSIC 조합도 실제 ZIP으로 실행합니다. 추가 학습이나 새 평가를 이용한 모델 선택은 하지 않습니다. 이번 할당은 최대 30분입니다.

In [ ]:
import time, json, shutil, subprocess, sys, zipfile, hashlib, threading, traceback, os
from pathlib import Path
from google.colab import drive, runtime
SESSION_START = time.time()
DEADLINE = SESSION_START + 1680
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/DACON_236749')
PRIMARY = DRIVE / 'runs_v2/20260912_174330'
RUN = PRIMARY / 'replay_verified'
RUN.mkdir(exist_ok=True)
WORK = Path('/content/dacon236749_replay')
WORK.mkdir(exist_ok=True)
(RUN / 'budget.json').write_text(json.dumps({'start_unix':SESSION_START,'deadline':DEADLINE,
    'first_allocation_elapsed_seconds':812.4415035247803,'second_allocation_limit_seconds':1800}))
watchdog = threading.Timer(max(1,DEADLINE-time.time()),runtime.unassign)
watchdog.daemon = True
watchdog.start()
for source in (PRIMARY / 'code').iterdir():
    if source.is_file():
        shutil.copy2(source,WORK / source.name)
print('REPLAY_RESULTS',RUN,flush=True)

In [ ]:
%%writefile /content/dacon236749_replay/package_hybrid_submission.py
"""Package the dev-locked G0/AST hybrid without changing either model's inference."""

from __future__ import annotations

import json
import math
import shutil
import zipfile
from pathlib import Path, PurePosixPath
from typing import Final

SUBMISSION_COLUMNS: Final = (
    "ID",
    "FILE_FAKE_PROB",
    "VOICE_FAKE_PROB",
    "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB",
    "MUSIC_PRESENT_PROB",
)
AST_COLUMNS: Final = ("MUSIC_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB")
RUNTIME_FILES: Final = ("ast_multitask.py", "v2_inference.py", "package_hybrid_submission.py")
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run the dev-locked G0/AST hybrid completely offline."""
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "model" / "runtime"))

from hybrid_runtime import main

if __name__ == "__main__":
    main(ROOT)
'''
HYBRID_RUNTIME: Final = '''"""Execute frozen G0 and AST, then route only same-ID probabilities."""
import csv
from pathlib import Path

import g0_runtime
from package_hybrid_submission import SUBMISSION_COLUMNS, route_predictions
from v2_inference import generate_submission


def _read_rows(path: Path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != list(SUBMISSION_COLUMNS):
            raise ValueError("Component submission columns differ from the official schema")
        return list(reader)


def main(root: Path):
    output = root / "output"
    output.mkdir(parents=True, exist_ok=True)
    g0_runtime.main()
    g0_path = output / "submission.csv"
    staged_g0 = output / "g0_submission.csv"
    g0_path.replace(staged_g0)
    ast_path = generate_submission(root)
    rows = route_predictions(_read_rows(staged_g0), _read_rows(ast_path))
    temporary = ast_path.with_suffix(".csv.tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(ast_path)
'''


def _validated_rows(rows: list[dict[str, str]], name: str) -> tuple[list[str], dict[str, dict[str, float]]]:
    """Parse one component CSV once and reject malformed or duplicate evaluator IDs."""
    order: list[str] = []
    parsed: dict[str, dict[str, float]] = {}
    for row in rows:
        if set(row) != set(SUBMISSION_COLUMNS):
            raise ValueError(f"{name} submission columns differ from the official schema")
        identifier = row["ID"]
        if not identifier or identifier in parsed:
            raise ValueError(f"{name} submission IDs must be non-empty and unique")
        try:
            probabilities = {column: float(row[column]) for column in SUBMISSION_COLUMNS[1:]}
        except ValueError as error:
            raise ValueError(f"{name} submission has a non-numeric probability") from error
        if any(not math.isfinite(value) or value < 0.0 or value > 1.0 for value in probabilities.values()):
            raise ValueError(f"{name} submission has an invalid probability")
        order.append(identifier)
        parsed[identifier] = probabilities
    if not order:
        raise ValueError(f"{name} submission is empty")
    return order, parsed


def route_predictions(g0_rows: list[dict[str, str]], ast_rows: list[dict[str, str]]) -> list[dict[str, float | str]]:
    """Route each ID's two frozen G0 values and three frozen AST values without mixing files."""
    ast_order, ast = _validated_rows(ast_rows, "AST")
    _g0_order, g0 = _validated_rows(g0_rows, "G0")
    if set(ast) != set(g0):
        raise ValueError("G0 and AST submission IDs differ")
    return [
        {
            "ID": identifier,
            "FILE_FAKE_PROB": g0[identifier]["FILE_FAKE_PROB"],
            "VOICE_FAKE_PROB": g0[identifier]["VOICE_FAKE_PROB"],
            "MUSIC_FAKE_PROB": ast[identifier]["MUSIC_FAKE_PROB"],
            "VOICE_PRESENT_PROB": ast[identifier]["VOICE_PRESENT_PROB"],
            "MUSIC_PRESENT_PROB": ast[identifier]["MUSIC_PRESENT_PROB"],
        }
        for identifier in ast_order
    ]


def _safe_members(archive: zipfile.ZipFile) -> list[zipfile.ZipInfo]:
    """Keep only the already-validated G0 model tree and reject path collisions."""
    members = [member for member in archive.infolist() if member.filename.startswith("model/")]
    names = [member.filename for member in members]
    if len(names) != len(set(names)) or any(name.startswith("model/ast/") for name in names):
        raise ValueError("G0 archive model members are unsafe for hybrid packaging")
    for member in members:
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
            raise ValueError(f"Unsafe G0 archive member: {member.filename}")
    return members


def build_hybrid_submission(original_zip: Path, ast_checkpoint_dir: Path, output_zip: Path) -> Path:
    """Build the fixed G0/AST hybrid from the immutable baseline and selected AST checkpoint."""
    from package_submission import MAX_EXPANDED_BYTES, MAX_ZIP_BYTES, build_submission
    from package_v2_submission import MODEL_INFO, _checkpoint_members

    checkpoint = _checkpoint_members(ast_checkpoint_dir.resolve())
    output_zip = output_zip.resolve()
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    g0_archive = output_zip.with_name(output_zip.name + ".g0.tmp")
    temporary = output_zip.with_name(output_zip.name + ".tmp")
    try:
        build_submission(original_zip, g0_archive, {"graph": "G0", "fusion": None})
        with zipfile.ZipFile(g0_archive) as source:
            members = _safe_members(source)
            g0_script = source.read("script.py").decode("utf-8")
            root_line = "ROOT = Path(__file__).resolve().parent"
            if g0_script.count(root_line) != 1:
                raise ValueError("G0 runtime root contract differs from the expected package")
            g0_runtime = g0_script.replace(root_line, "ROOT = Path(__file__).resolve().parents[2]", 1)
            compile(g0_runtime, "model/runtime/g0_runtime.py", "exec")
            source_root = Path(__file__).resolve().parent
            runtime = {name: source_root / name for name in RUNTIME_FILES}
            license_text = (source_root / "AST_LICENSE.txt").read_bytes()
            if any(not path.is_file() for path in runtime.values()):
                raise ValueError("Hybrid runtime source is incomplete")
            expanded = sum(member.file_size for member in members)
            expanded += sum(path.stat().st_size for path in checkpoint.values())
            expanded += sum(path.stat().st_size for path in runtime.values()) + len(g0_runtime.encode())
            expanded += len(DEPLOY_SCRIPT.encode()) + len(HYBRID_RUNTIME.encode()) + len(json.dumps(MODEL_INFO).encode()) + len(license_text)
            if expanded > MAX_EXPANDED_BYTES or expanded > MAX_ZIP_BYTES:
                raise ValueError(f"Hybrid archive would exceed submission limits: {expanded} bytes")
            with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as target:
                for member in members:
                    if member.is_dir():
                        target.writestr(member.filename, b"")
                    else:
                        with source.open(member) as incoming, target.open(member.filename, "w", force_zip64=True) as outgoing:
                            shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)
                for name, path in checkpoint.items():
                    target.write(path, f"model/ast/{name}")
                target.writestr("model/ast/MODEL_INFO.json", json.dumps(MODEL_INFO, sort_keys=True))
                target.writestr("model/ast/LICENSE.txt", license_text)
                for name, path in runtime.items():
                    target.write(path, f"model/runtime/{name}")
                target.writestr("model/runtime/g0_runtime.py", g0_runtime)
                target.writestr("model/runtime/hybrid_runtime.py", HYBRID_RUNTIME)
                target.writestr("script.py", DEPLOY_SCRIPT)
                target.writestr("requirements.txt", "")
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            raise ValueError("Built hybrid archive exceeds compressed submission size limit")
        temporary.replace(output_zip)
    finally:
        temporary.unlink(missing_ok=True)
        g0_archive.unlink(missing_ok=True)
    return output_zip


In [ ]:
%%writefile /content/dacon236749_replay/verify_v2_gpu_precision.py
"""Diagnose A100 package parity without changing the 1e-5 acceptance limit."""

from __future__ import annotations

import argparse
import csv
import gc
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch

from ast_multitask import PREDICTION_COLUMNS, load_checkpoint
from graph_features import atomic_json, digest
from package_v2_submission import CHECKPOINT_FILES
from v2_inference import ordered_audio_files, predict_audio_file


def _rows(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def _reference(checkpoint: Path, inputs: Path, output: Path, deadline: float) -> list[dict]:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for the GPU precision diagnostic")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    mode = json.loads((checkpoint / "multitask_config.json").read_text())["mode"]
    model, processor = load_checkpoint(checkpoint, mode)
    model = model.cuda().eval()
    requested = [row["ID"] for row in _rows(inputs / "sample_submission.csv")]
    paths = ordered_audio_files(inputs / "test", requested)
    records = []
    for path in paths:
        if time.time() >= deadline:
            raise TimeoutError("reference prediction deadline reached")
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()
        probabilities = predict_audio_file(model, processor, path)
        torch.cuda.synchronize()
        records.append({"ID": path.stem, **dict(zip(PREDICTION_COLUMNS, probabilities, strict=True)),
                        "seconds": time.perf_counter() - started,
                        "max_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
                        "max_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3})
    with output.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return records


def _child_code(cudnn: bool | None, matmul: bool | None) -> str:
    assignments = []
    if cudnn is not None:
        assignments.append(f"torch.backends.cudnn.allow_tf32={cudnn}")
    if matmul is not None:
        assignments.append(f"torch.backends.cuda.matmul.allow_tf32={matmul}")
    return ";".join([
        "import json,runpy,torch,time",
        *assignments,
        "torch.cuda.reset_peak_memory_stats()",
        "started=time.perf_counter()",
        "runpy.run_path('script.py',run_name='__main__')",
        "torch.cuda.synchronize()",
        "print('GPU_DIAGNOSTIC '+json.dumps({'cudnn_tf32':torch.backends.cudnn.allow_tf32,'matmul_tf32':torch.backends.cuda.matmul.allow_tf32,'seconds':time.perf_counter()-started,'max_allocated_gib':torch.cuda.max_memory_allocated()/1024**3,'max_reserved_gib':torch.cuda.max_memory_reserved()/1024**3}))",
    ])


def _run_variant(name: str, flags: tuple[bool | None, bool | None], root: Path,
                 work: Path, deadline: float) -> tuple[list[dict[str, str]], dict]:
    remaining = deadline - time.time()
    if remaining <= 0:
        raise TimeoutError(f"deadline reached before {name}")
    result = subprocess.run(
        [sys.executable, "-c", _child_code(*flags)], cwd=root,
        env=os.environ | {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"},
        text=True, capture_output=True, timeout=remaining, check=False,
    )
    (work / f"{name}.log").write_text(result.stdout + result.stderr, encoding="utf-8")
    result.check_returncode()
    receipt = json.loads(next(line.removeprefix("GPU_DIAGNOSTIC ") for line in result.stdout.splitlines()
                              if line.startswith("GPU_DIAGNOSTIC ")))
    destination = work / f"{name}.csv"
    shutil.copy2(root / "output" / "submission.csv", destination)
    return _rows(destination), receipt


def _classify(variants: dict) -> tuple[str, dict]:
    delta = {name: value["max_delta"] for name, value in variants.items()}
    default = variants["default"]
    precision_state = delta["default"] > 1e-5 and delta["both_false"] <= 1e-5
    cudnn_signature = (default["cudnn_tf32"] is True and default["matmul_tf32"] is False
                       and delta["cudnn_true_matmul_false"] > 1e-5)
    diagnosis = ("confirmed_cudnn_tf32" if precision_state and cudnn_signature
                 else "confirmed_process_precision_state" if precision_state else "inconclusive")
    return diagnosis, {"cudnn_false_matmul_true_max_delta": delta["cudnn_false_matmul_true"],
                       "passes_1e-5": delta["cudnn_false_matmul_true"] <= 1e-5}


def main() -> None:
    parser = argparse.ArgumentParser()
    for name in ("checkpoint", "package", "inputs", "work", "output"):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--deadline", type=float, required=True)
    args = parser.parse_args()
    args.work.mkdir(parents=True, exist_ok=True)
    reference = _reference(args.checkpoint, args.inputs, args.work / "reference.csv", args.deadline)
    unpacked = args.work / "unpacked"
    unpacked.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(args.package) as archive:
        if any(Path(name).is_absolute() or ".." in Path(name).parts for name in archive.namelist()):
            raise ValueError("unsafe package member")
        archive.extractall(unpacked)
    shutil.copytree(args.inputs, unpacked / "data", dirs_exist_ok=True)
    hashes = {name: {"reference": digest(args.checkpoint / name),
                     "package": digest(unpacked / "model" / "ast" / name)} for name in CHECKPOINT_FILES}
    if any(value["reference"] != value["package"] for value in hashes.values()):
        raise ValueError("package checkpoint assets differ from reference")
    variants = {"default": (None, None), "both_false": (False, False),
                "cudnn_false_matmul_true": (False, True),
                "cudnn_true_matmul_false": (True, False)}
    report = {"asset_sha256": hashes, "reference": reference, "variants": {}}
    for name, flags in variants.items():
        rows, receipt = _run_variant(name, flags, unpacked, args.work, args.deadline)
        by_id = {row["ID"]: row for row in rows}
        deltas = {row["ID"]: max(abs(float(row[column]) - float(by_id[row["ID"]][column]))
                                      for column in PREDICTION_COLUMNS) for row in reference}
        report["variants"][name] = {**receipt, "per_file_max_delta": deltas,
                                     "max_delta": max(deltas.values()), "csv": str(args.work / f"{name}.csv")}
        atomic_json(args.output, report)
    delta = {name: value["max_delta"] for name, value in report["variants"].items()}
    report["diagnosis"], report["secondary_sensitivity"] = _classify(report["variants"])
    report["threshold"] = 1e-5
    atomic_json(args.output, report)
    print(json.dumps({"diagnosis": report["diagnosis"], "deltas": delta}, sort_keys=True), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
try:
    packages = ['demucs==4.0.1','panns-inference==0.1.1','transformers==4.57.6',
                'librosa==0.10.2.post1','diffq==0.2.4','einops==0.8.1',
                'scikit-learn==1.8.0','soundfile==0.12.1','safetensors']
    with (RUN / 'install.log').open('w') as log:
        subprocess.run([sys.executable,'-m','pip','install','-q',*packages],
            stdout=log,stderr=subprocess.STDOUT,check=True,timeout=600)
    checkpoint = WORK / 'checkpoint'
    shutil.copytree(PRIMARY / 'training/AST_LAST_BLOCK/best',checkpoint,dirs_exist_ok=True)
    inputs = WORK / 'inputs'
    shutil.copytree(PRIMARY / 'package_inputs',inputs,dirs_exist_ok=True)
    old_package = WORK / 'old_AST_LAST_BLOCK_submission.zip'
    shutil.copy2(PRIMARY / 'AST_LAST_BLOCK_submission.zip',old_package)
    os.chdir(WORK)
    sys.path.insert(0,str(WORK))
    from graph_features import atomic_json, digest
    from package_submission import verify_packaged
    from package_v2_submission import build_ast_submission
    from package_hybrid_submission import build_hybrid_submission, route_predictions
    from run_experiment import read_csv, write_csv
    from graph_scoring import SUBMISSION_COLUMNS
    lock = json.loads((PRIMARY / 'hybrid_candidate_lock.json').read_text())
    assert lock['selected'] == 'G0_AST_MUSIC'
    assert digest(PRIMARY / 'hybrid_selection.json') == lock['selection_sha256']
    for name, expected in lock['checkpoints']['AST_LAST_BLOCK'].items():
        assert digest(checkpoint / name) == expected, f'Checkpoint changed: {name}'
    diagnostic = WORK / 'precision'
    cmd = [sys.executable,'-u','verify_v2_gpu_precision.py','--checkpoint',str(checkpoint),
           '--package',str(old_package),'--inputs',str(inputs),'--work',str(diagnostic),
           '--output',str(RUN / 'precision_diagnostic.json'),'--deadline',str(DEADLINE-600)]
    with (RUN / 'precision_run.log').open('w') as log:
        subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT,check=True,timeout=max(1,DEADLINE-time.time()-600))
    diagnosis = json.loads((RUN / 'precision_diagnostic.json').read_text())
    for source in diagnostic.iterdir():
        if source.is_file():
            shutil.copy2(source,RUN / source.name)
    print('PRECISION_DIAGNOSIS',diagnosis['diagnosis'],diagnosis['variants'],flush=True)
    assert diagnosis['diagnosis'] in ('confirmed_cudnn_tf32','confirmed_process_precision_state'), 'Precision cause not established'
    source_path = WORK / 'v2_inference.py'
    before = source_path.read_text()
    anchor = '    checkpoint = root / "model" / "ast"'
    assert before.count(anchor) == 1
    fixed = before.replace(anchor,'    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n'+anchor,1)
    source_path.write_text(fixed)
    compile(fixed,'v2_inference.py','exec')
    atomic_json(RUN / 'precision_fix.json',{'reason':diagnosis['diagnosis'],
        'before_sha256':hashlib.sha256(before.encode()).hexdigest(),
        'after_sha256':digest(source_path),'threshold':1e-5,'weights_changed':False})
    with zipfile.ZipFile(DRIVE / 'open.zip') as archive:
        member = next(n for n in archive.namelist() if n.endswith('baseline_submit.zip'))
        original = Path(archive.extract(member,WORK / 'original'))
    assert digest(original) == lock['original_zip_sha256']
    baseline = WORK / 'baseline'
    with zipfile.ZipFile(original) as archive:
        archive.extractall(baseline)
    from run_v2 import baseline_predictions
    input_rows = [{'ID':r['ID'],'path':'test/'+r['ID']+'.wav'} for r in read_csv(inputs / 'sample_submission.csv')]
    g0 = baseline_predictions(baseline,input_rows,inputs,RUN,DEADLINE-300)
    ast_reference = [{c:r[c] for c in SUBMISSION_COLUMNS} for r in read_csv(diagnostic / 'reference.csv')]
    expected = {'AST_LAST_BLOCK':ast_reference,'G0_AST_MUSIC':route_predictions(g0,ast_reference)}
    guard = WORK / 'network_guard'
    guard.mkdir(exist_ok=True)
    (guard / 'sitecustomize.py').write_text('import socket\ndef deny(*args, **kwargs):\n    raise RuntimeError("network forbidden in offline replay")\nsocket.socket.connect = deny\nsocket.create_connection = deny\n')
    receipts = {}
    for name in expected:
        package = WORK / (name+'_submission.zip')
        if name == 'AST_LAST_BLOCK':
            build_ast_submission(checkpoint,package)
        else:
            build_hybrid_submission(original,checkpoint,package)
        unpacked = WORK / ('verified_'+name)
        with zipfile.ZipFile(package) as archive:
            expanded = sum(x.file_size for x in archive.infolist())
            archive.extractall(unpacked)
        shutil.copytree(inputs,unpacked / 'data',dirs_exist_ok=True)
        old_pythonpath = os.environ.get('PYTHONPATH')
        os.environ['PYTHONPATH'] = str(guard)
        started = time.perf_counter()
        try:
            predicted = verify_packaged(unpacked,DEADLINE-120)
        finally:
            if old_pythonpath is None:
                os.environ.pop('PYTHONPATH',None)
            else:
                os.environ['PYTHONPATH'] = old_pythonpath
        seconds = time.perf_counter()-started
        assert [r['ID'] for r in predicted] == [r['ID'] for r in expected[name]]
        delta = max(abs(float(a[c])-float(b[c])) for a,b in zip(predicted,expected[name],strict=True) for c in SUBMISSION_COLUMNS[1:])
        assert delta <= 1e-5, f'Package parity: {name} {delta}'
        shutil.copy2(package,RUN / package.name)
        assert digest(package) == digest(RUN / package.name)
        shutil.copy2(unpacked / 'output/packaged_inference.log',RUN / (name+'_packaged.log'))
        write_csv(RUN / (name+'_reference.csv'),expected[name])
        write_csv(RUN / (name+'_packaged_predictions.csv'),predicted)
        receipts[name] = {'max_probability_delta':delta,'files':len(predicted),
            'script_seconds_including_load':seconds,'zip_bytes':package.stat().st_size,
            'expanded_bytes':expanded,'zip_sha256':digest(package),
            'network_socket_guard':True,'stress_60s_included':True,'L4_runtime_limit':'unverified'}
        atomic_json(RUN / 'packaged_verification.json',receipts)
        print('PACKAGE_VERIFIED',name,receipts[name],flush=True)
        shutil.rmtree(unpacked)
    (RUN / 'code').mkdir(exist_ok=True)
    for source in WORK.glob('*.py'):
        shutil.copy2(source,RUN / 'code' / source.name)
    atomic_json(RUN / 'finished.json',{'status':'finished','selected':'G0_AST_MUSIC',
        'elapsed_seconds':time.time()-SESSION_START,'submission_status':'not_submitted'})
    print('REPLAY_SAVED',RUN,flush=True)
except Exception:
    (RUN / 'failure.txt').write_text(traceback.format_exc())
    print(traceback.format_exc(),flush=True)
    raise
finally:
    (RUN / 'release_requested.json').write_text(json.dumps({'unix':time.time(),
        'elapsed_seconds':time.time()-SESSION_START}))
    watchdog.cancel()
    runtime.unassign()